# Layered-earth calibration and interval coverage

**Level 2 — Applied analysis**  
**Prerequisites:** Level 1 and arrays/logarithms; 35 min.

## Objectives

Trace the evidence from source contract to result. State units and assumptions, reproduce the calculation, and identify which inference the evidence cannot support.

## Setup

Uses repository modules and committed records through a discovered root; it downloads and mutates nothing. Install exact pins: `pip install numpy==2.2.6 matplotlib==3.11.2` `scipy==1.15.2` is required by the MT forward operator.

## Data

Measured QC-only record plus a separate seeded synthetic calibration artifact; never merge these evidence classes.

## Method

This artifact is explicitly synthetic: a 1D two-layer earth, `[180,20] Ω·m`, upper thickness 450 m, 24 frequencies, 48 independent noisy realizations and 128 bootstrap draws per realization. Empirical interval coverage is `C=N⁻¹Σ1[ρtrue∈CI]`; the independent unit is the realization, not each bootstrap draw. Compare coverage with the 95% nominal level and stored Wilson bounds, then inspect interval width in Ω·m. The interval is conditional on fixed thickness, independent Gaussian real/imaginary errors, bounds and regularization. It is not a field confidence guarantee; high apparent coverage can coexist with biased or weakly resolved layers.

Parameters are log-resistivities: positive by construction, with equal steps representing equal multiplicative changes. Complex residuals use per-real-component sigma; real and imaginary parts both inform the fit. Frequencies are not independent depth samples: sensitivity kernels overlap, so one period does not identify one interface. Local sensitivity and coverage remain conditional on the fixed parameterization and cannot rule out global equivalence.

## Results

Read the physical axes and units; distinguish data fit, error against synthetic truth and agreement with analyst labels. Recomputed values must agree with the serialized receipt. A plotted model alone does not establish uniqueness, field transfer or resolution.

### Diagnostic 1

Inspect the committed case record and provenance.

In [ ]:
from pathlib import Path
from importlib.metadata import version
print("Runtime versions:", {name: version(name) for name in ['numpy', 'matplotlib', 'scipy']})
import numpy as np
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/"data").is_dir() and (p/"data-pipeline").is_dir())
# Inspect synthetic calibration provenance and independent realization rows.
import json
from pathlib import Path
cal=json.loads((root/"data/derived/v2/edi/calibration-two-layer.json").read_text(encoding="utf-8"))
print(cal["kind"],cal["calibration_model"],"Ω·m; thickness",cal["fixed_thickness_m"],"m; frequencies",len(cal["frequencies_hz"]))
print("realizations",cal["realizations"],"bootstrap each",cal["bootstrap_samples_per_realization"],"nominal",cal["confidence"],"coverage",cal["coverage_per_layer"])

### Diagnostic 2

Compute the declared comparison and plot it with physical units.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# Recompute coverage and median interval widths from the 48 recorded fits.
import matplotlib.pyplot as plt
rows=cal["rows"]; truth=np.asarray(cal["calibration_model"],float); cover=np.asarray([r["covered"] for r in rows],bool); ci=np.asarray([r["interval_ohm_m"] for r in rows],float)
cov=cover.mean(0); width=np.median(ci[:,1,:]-ci[:,0,:],axis=0); print("coverage",cov,"; median interval width Ω·m",width)
fig,ax=plt.subplots(1,2,figsize=(8,3.5),constrained_layout=True); x=np.arange(2); ax[0].bar(x,cov); ax[0].axhline(cal["confidence"],ls="--",color="crimson"); ax[0].errorbar(x,cov,yerr=[np.maximum(0,cov-np.asarray(cal["coverage_wilson95_lower"])),np.maximum(0,np.asarray(cal["coverage_wilson95_upper"])-cov)],fmt="none",capsize=4,color="black"); ax[0].set(xticks=x,xticklabels=["upper","lower"],ylim=(0,1.05),ylabel="Coverage fraction",title="Realization-level coverage")
ax[1].bar(x,width,color="#e1812c"); ax[1].set(xticks=x,xticklabels=["upper","lower"],ylabel="Median interval width (Ω·m)",title="Conditional uncertainty"); [a.grid(axis="y",alpha=.2) for a in ax]; plt.show()

## Summary

Conclusion is conditional on geometry, acquisition, noise model, split, parameterization and source rights. Preserve negative cases and name the next observation or control needed.

## References

- [https://github.com/MTgeophysics/mtpy-v2](https://github.com/MTgeophysics/mtpy-v2)
- [https://www.usgs.gov/data/magnetotelluric-data-clear-lake-region-northern-california](https://www.usgs.gov/data/magnetotelluric-data-clear-lake-region-northern-california)